# Viz comparison: real graph against v2 non-private and graph-aim-walks

The same plots as `viz_comparison.ipynb`, for the question of `docs/snap/walks-plan.md`: with privacy set aside, can
a generator that only sees readable tables of random-walk statistics produce graphs whose walks look real? Black is
always the **real** graph. The two synthetic series are both **non-private** (exact statistics of the real graph,
not a DP release): **v2 non-private** (graph-AIM v2, no caps, no noise; `snap/outside/synthetic_v2/`) and
**graph-aim-walks** (`snap/outside/synthetic_walks/`), which starts from v2 and then matches the walk tables.

Sections 1-6 and 8 are the plots of `viz_comparison.ipynb` unchanged. Section 9 shows communities and section 10 the
walk statistics, both **matched** (what the walks generator was fitted to: walks of up to 32 steps) and **held out**
(never matched: walks beyond 32 steps, windows of 64 and 128 steps, Louvain communities). The held-out checks are
the ones that say whether the generator learned the structure or only the tables. Section 11 gives the numbers.
A series whose graph file does not exist yet is skipped (the setup cell says which).

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

NB = Path.cwd()
EVAL = NB.parent if NB.name == "notebooks" else NB          # snap/eval
SNAP = EVAL.parent
sys.path.insert(0, str(SNAP.parent / "src"))

from graph_aim.eval import viz
from graph_aim.eval.evaluate import align_columns, comparison_table, degrees
from graph_aim.transfer.graph_io import load_attributed_graph
from graph_aim_walks.eval import viz as walks_viz
from graph_aim_walks.eval.metrics import community_metrics, default_spec, walk_metrics, walk_profile

from IPython.display import display

plt.rcParams["figure.dpi"] = 110


def show(fig):
    # display explicitly (works on any backend), then free the figure
    display(fig)
    plt.close(fig)

real = load_attributed_graph(SNAP / "inside" / "data" / "processed" / "attributed_graph.npz")
ATTR = json.loads((SNAP / "inside" / "config" / "graph_aim_v1.json").read_text())["homophily_attrs"][0]

# label -> graph written by stage 2 (seed 0). A missing file is skipped, not an error.
SYNTHETIC = {
    "v2 non-private": SNAP / "outside" / "synthetic_v2" / "graph_aim_nocap_nonprivate_seed0.npz",
    "graph-aim-walks": SNAP / "outside" / "synthetic_walks" / "graph_aim_walks_seed0.npz",
}
syns = {label: align_columns(load_attributed_graph(path), real) for label, path in SYNTHETIC.items() if path.exists()}
for label, path in SYNTHETIC.items():
    if not path.exists():
        print(f"{label}: not yet generated ({path.relative_to(SNAP)} is missing), skipped")
for label, g in {"real": real, **syns}.items():
    print(f"{label:24s} nodes {g.n}  edges {len(g.edges):6d}  mean degree {2 * len(g.edges) / g.n:5.1f}  max degree {degrees(g).max()}")
print("attribute used for colours and homophily:", ATTR)

## 1. Degrees

Left: the share of nodes with at least *k* friends (log-log). Right: every node's degree sorted from the biggest hub
down. *Look for:* graph-aim-walks keeps the degree sequence of its v2 start exactly (its structure moves are
degree-preserving swaps), so the two coloured lines should coincide.

In [ ]:
show(viz.degree_distribution(real, syns))

**What we see.** _To be written after execution._

## 2. Clustering: do friends of friends know each other?

Left: average local clustering of nodes with the same degree. Right: the spread of local clustering over all nodes.
*Look for:* the walks generator keeps `clustering_by_degree` in its energy; does it hold the level while it rewires?

In [ ]:
show(viz.clustering_by_degree(real, syns))

**What we see.** _To be written after execution._

## 3. Who connects to whom, by degree

Each heatmap row is a degree class; the colours show where that class's friends sit (also by degree). The title gives
the assortativity coefficient. The curve below shows the average degree of a person's friends against their own
degree. *Look for:* the walks' `degree_lag` table is degree mixing at walk scale; does it move the pattern?

In [ ]:
show(viz.degree_mixing(real, syns))

In [ ]:
show(viz.neighbor_degree(real, syns))

**What we see.** _To be written after execution._

## 4. Distances and dense cores (held out)

Left: how many hops separate two random people. Right: how many nodes belong to a dense core. Neither is matched by
the walks generator (path lengths and k-cores are held out in the plan).

In [ ]:
show(viz.paths_and_cores(real, syns))

**What we see.** _To be written after execution._

## 5. Attributes

**5a. How common is each value.** The bars compare the share of nodes per value; the walks generator's label moves
swap values between nodes, so these shares are those of its v2 start.

In [ ]:
show(viz.attribute_marginals(real, syns))

**What we see.** _To be written after execution._

**5b. Do friends share attributes (homophily).** For people in a group of `ATTR` (rows), where do their friends
fall (columns)? Numbers are percent of friendships; a strong diagonal means friends tend to be in the same group.

In [ ]:
show(viz.homophily(real, syns, ATTR))

**What we see.** _To be written after execution._

**5c. Do attributes go together.** Cramer's V between each pair of node attributes in the real graph, in the
synthetic graph, and their difference (orange = synthetic more associated, blue = less).

In [ ]:
show(viz.attribute_association(real, syns))

**What we see.** _To be written after execution._

## 6. Small examples

The same selection rule for every graph: the node whose degree is closest to the target, with median clustering
among those. Typical, not cherry-picked.

**6a. One person and their friends**, coloured by `ATTR`; columns are increasing degree.

In [ ]:
show(viz.ego_examples(real, syns, ATTR, targets=(12, 30, 70)))

**What we see.** _To be written after execution._

**6b. A connected patch of 60 nodes** found by walking outward from a typical node.

In [ ]:
show(viz.bfs_sample(real, syns, ATTR, size=60, start_degree=20))

**What we see.** _To be written after execution._

## 8. Attribute coupling

Which kinds of people have many connections, and which kinds connect to each other (definitions in
`src/graph_aim/eval/coupling.py`). Mean degree per value, same-value lift per attribute against chance, and eta^2 of
log degree on each attribute.

In [ ]:
show(viz.degree_by_attribute(real, syns, ["school", "birthday"]))

In [ ]:
show(viz.homophily_lift(real, syns))

In [ ]:
show(viz.degree_attribute_coupling(real, syns))

**What we see.** _To be written after execution._

## 9. Communities (held out)

How the graph splits into groups of people who mostly know each other: Louvain (seed 0) and label propagation,
definitions in `src/graph_aim/eval/community.py` (those of `docs/snap/v2-diagnostic.md` section 1). (a) modularity;
(b) for every node, the size of its Louvain part; (c) for every node, the conductance of its part (0 = closed);
(d) the share of nodes in parts of at least 100 people, with the number of parts. The walks generator never sees a
partition. *Look for:* the real graph has a few large, closed parts (16 parts, node-weighted conductance 0.04); v2
reaches a similar modularity with many small, leaky ones. Does matching walks move the coloured curves toward black?

In [ ]:
show(viz.community_structure(real, syns))

**What we see.** _To be written after execution._

## 10. Walk statistics: matched and held out

One pool of 50,000 random walks of 128 steps per graph, with the contract's walk process
(`src/graph_aim_walks/transfer/walkstats.py`); statistics in `src/graph_aim_walks/eval/metrics.py`. The grey band is
held out: the generator matched walks of 32 steps (return, coverage, attribute agreement at lags up to 16, share
distributions in windows of 8 / 16 / 32 steps), never steps beyond 32, lags beyond 16, or windows of 64 and 128.
Hollow markers are held-out points.

- Row 1: probability that a walk is back where it started after *t* steps (dense regions trap walks); distinct nodes
  visited (how fast walks escape); the top 16 eigenvalues of the walk matrix (eigenvalues near 1 = near-separate
  regions); the across-walk variance of a value's share among the nodes a walk visits (high = walks stay in regions
  where one value dominates), averaged over school, hometown and location.
- Row 2: probability that the node *k* steps later is in the same group, per group.
- Row 3: the share variance per attribute.

*Look for:* how close the walks generator is on the matched part (it was fitted there), and above all whether the
held-out windows 64 and 128 follow: a generator that only learned the tables would fall away from black beyond the
band's edge.

In [ ]:
SPEC = default_spec(real)
profiles = {"real": walk_profile(real, SPEC, n_walks=50000, seed=0)}
profiles.update({label: walk_profile(g, SPEC, n_walks=50000, seed=0) for label, g in syns.items()})
show(walks_viz.walk_profiles(real, syns, spec=SPEC, profiles=profiles))

**What we see.** _To be written after execution._

## 11. The numbers

Distances to the real graph (0 = identical on that statistic), the community scalars of every graph, and the
walk-statistic gaps (`wlk_*`; share-variance ratio syn / real, 1 = real, per window with 64 and 128 held out). The
multi-seed versions are the `com_*` and `wlk_*` columns of `snap/eval/results_walks/sweep.csv` once scored.

In [ ]:
comparison_table(real, syns, homophily_attrs=[ATTR]).round(3)

In [ ]:
pd.DataFrame({label: community_metrics(g) for label, g in {"real": real, **syns}.items()}).round(4)

In [ ]:
pd.DataFrame({label: walk_metrics(real, g, real_profile=profiles["real"], syn_profile=profiles[label])
              for label, g in syns.items()}).round(4)

**What we see.** _To be written after execution._

## Caveats

- One synthetic graph per method (seed 0); the score sweep has the seed spread.
- Both synthetic series are **non-private**: their artifacts hold exact statistics of the real graph. They answer a
  fidelity question (can this generator class reach real-looking walks?), not a privacy one.
- The walk statistics of section 10 come from one pool of 50,000 walks per graph (seed 0): they carry sampling
  error, so small differences between curves are not findings.
- These plots compare distributions, not individual nodes: synthetic nodes are not matched to real ones.